# 🚀 ABSA Project - Full Training Pipeline

**Huấn luyện đầy đủ 6 mô hình ABSA trên Kaggle GPU (T4)**

| # | Mô hình | Kiểu | Base Model | Ước lượng |
|---|---------|------|------------|-----------|
| 1 | ViSoBERT-MTL | Multi-Task | visobert | ~50 phút |
| 2 | ViSoBERT-STL | Single-Task (2 stage) | visobert | ~60 phút |
| 3 | PhoBERT-MTL | Multi-Task | phobert-base | ~50 phút |
| 4 | PhoBERT-STL | Single-Task (2 stage) | phobert-base | ~60 phút |
| 5 | BiLSTM-MTL | Multi-Task | PhoBERT embeddings | ~40 phút |
| 6 | BiLSTM-STL | Single-Task (2 stage) | PhoBERT embeddings | ~40 phút |

**Tổng thời gian ước tính: ~5-6 giờ** (có thể ít hơn với early stopping)

> **Unified Configuration:** seed=42, AdamW optimizer, Focal Loss (γ=2.0), max_length=256, fp16=True

---
## 📦 Bước 1: Dọn dẹp workspace

In [ ]:
!rm -rf /kaggle/working/*

---
## 🐍 Bước 2: Cài đặt Miniconda + Python 3.10

In [ ]:
# Cài đặt Miniconda vào thư mục có quyền ghi
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
!bash /tmp/miniconda.sh -b -p /kaggle/working/miniconda -f
!rm /tmp/miniconda.sh

# Tạo environment Python 3.10
!/kaggle/working/miniconda/bin/conda create -y -n absa python=3.10

# Verify
!/kaggle/working/miniconda/envs/absa/bin/python --version

---
## 📥 Bước 3: Clone repository & Kiểm tra dataset

In [ ]:
# Khai báo biến activate môi trường
ACTIVATE = "source /kaggle/working/miniconda/bin/activate absa"

# Clone repo
!git clone https://github.com/hungtran3028/ABSA-project.git /kaggle/working/ABSA-project

%cd /kaggle/working/ABSA-project

# Kiểm tra dataset
!wc -l data/UIT-ViSFD/*.csv
!echo "---"
!head -3 data/UIT-ViSFD/train.csv

---
## 📚 Bước 4: Cài đặt thư viện

In [ ]:
# Cài đặt PyTorch với CUDA 12.1 (tối ưu cho Kaggle T4)
!{ACTIVATE} && pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Cài đặt các thư viện cần thiết
!{ACTIVATE} && pip install transformers datasets accelerate wandb scikit-learn pandas numpy matplotlib seaborn underthesea pyvi emoji sentencepiece protobuf tqdm pyyaml statsmodels

In [ ]:
# Kiểm tra GPU & PyTorch
# Viết script tạm để tránh lỗi IPython variable expansion
with open('/tmp/check_gpu.py', 'w') as f:
    f.write('''
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
''')

!{ACTIVATE} && python /tmp/check_gpu.py

---
## 📊 Bước 4.5: Cấu hình Wandb Tracking

> ⚠️ **Yêu cầu**: Trước khi chạy, cần thêm Kaggle Secret:
> 1. Vào **Add-ons → Secrets**
> 2. Thêm key: `WANDB_API_KEY` với value là API key từ [wandb.ai](https://wandb.ai/authorize)

In [ ]:
import os

# Lấy API key từ Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = wandb_key
    print("✅ WANDB_API_KEY loaded from Kaggle Secrets")
except Exception as e:
    print(f"⚠️ Could not load WANDB_API_KEY: {e}")
    print("Wandb will run in offline mode")
    os.environ["WANDB_MODE"] = "offline"

# Login wandb
!{ACTIVATE} && wandb login --relogin $WANDB_API_KEY 2>/dev/null || echo "Wandb offline mode"

os.environ["WANDB_PROJECT"] = "ABSA-Vietnamese"
print(f"📊 Wandb project: {os.environ.get('WANDB_PROJECT', 'N/A')}")

---
## 🔄 Bước 5: Chuẩn bị & Cân bằng dữ liệu

Chạy pipeline chia dữ liệu (train/val/test 80/10/10) và oversampling cân bằng lớp cho Sentiment Classification.

In [ ]:
# Chạy pipeline chuẩn bị dữ liệu đầy đủ
!{ACTIVATE} && bash run_data_preparation.sh

In [ ]:
# Copy dữ liệu cho PhoBERT-STL và phoBERT-MTL 
# (script prepare_data chỉ lưu vào 4 thư mục: BILSTM-MTL, BILSTM-STL, VisoBERT-MTL, VisoBERT-STL)
import shutil, os

src_dirs = {
    "VisoBERT-MTL/data": "phoBERT-MTL/data",
    "VisoBERT-STL/data": "PhoBERT-STL/data",
}

for src, dst in src_dirs.items():
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"✅ Copied {src} → {dst}")
    else:
        print(f"⚠️ Source not found: {src}")

---
---
# 🏋️ HUẤN LUYỆN 6 MÔ HÌNH

> ⚠️ **Lưu ý**: Mỗi mô hình BERT cần ~30-50 phút, BiLSTM cần ~20-40 phút. Tổng ~4-6 giờ.
> Notebook sẽ tự động lưu kết quả sau mỗi mô hình (checkpointing).

## 🔵 Model 1/6: ViSoBERT-MTL (Multi-Task Learning)

- **Base model**: `5CD-AI/visobert-14gb-corpus`
- **Architecture**: ViSoBERT → Shared Encoder → 2 Task Heads (AD + SC)
- **Epochs**: 20 (early stopping patience 3) | **Batch size**: 16 | **LR**: 2e-5
- **Wandb run**: `ViSoBERT-MTL`
- **⏱️ ~50 phút**

In [ ]:
!{ACTIVATE} && MPLBACKEND=Agg python VisoBERT-MTL/train_visobert_mtl.py --config VisoBERT-MTL/config_visobert_mtl.yaml

In [ ]:
# Xem kết quả ViSoBERT-MTL
!cat VisoBERT-MTL/models/mtl/final_report.txt

---
## 🟢 Model 2/6: ViSoBERT-STL (Single-Task Learning - 2 Stage)

- **Base model**: `5CD-AI/visobert-14gb-corpus`  
- **Architecture**: ViSoBERT → Task-specific Heads (AD riêng, SC riêng)
- **Epochs**: AD 20, SC 20 (early stopping patience 3) | **Batch size**: 16 | **LR**: 2e-5
- **Wandb runs**: `ViSoBERT-STL-AD`, `ViSoBERT-STL-SC`
- **⏱️ ~60 phút** (2 stages)

In [ ]:
!{ACTIVATE} && MPLBACKEND=Agg python VisoBERT-STL/train_visobert_stl.py --config VisoBERT-STL/config_visobert_stl.yaml

In [ ]:
# Xem kết quả ViSoBERT-STL
!cat VisoBERT-STL/results/two_stage_training/final_report.txt

---
## 🔵 Model 3/6: PhoBERT-MTL (Multi-Task Learning)

- **Base model**: `vinai/phobert-base`
- **Architecture**: PhoBERT → Shared Encoder → 2 Task Heads (AD + SC)
- **Epochs**: 20 (early stopping patience 3) | **Batch size**: 16 | **LR**: 2e-5
- **Wandb run**: `PhoBERT-MTL`
- **⏱️ ~50 phút**

In [ ]:
!{ACTIVATE} && MPLBACKEND=Agg python phoBERT-MTL/train_phobert_mtl.py --config phoBERT-MTL/config_phobert_mtl.yaml

In [ ]:
# Xem kết quả PhoBERT-MTL
!cat phoBERT-MTL/models/mtl/final_report.txt

---
## 🟢 Model 4/6: PhoBERT-STL (Single-Task Learning - 2 Stage)

- **Base model**: `vinai/phobert-base`
- **Architecture**: PhoBERT → Task-specific Heads
- **Epochs**: AD 20, SC 20 (early stopping patience 3) | **Batch size**: 16 | **LR**: 2e-5
- **Wandb runs**: `PhoBERT-STL-AD`, `PhoBERT-STL-SC`
- **⏱️ ~60 phút** (2 stages)

In [ ]:
!{ACTIVATE} && MPLBACKEND=Agg python PhoBERT-STL/train_phobert_stl.py --config PhoBERT-STL/config_phobert_stl.yaml

In [ ]:
# Xem kết quả PhoBERT-STL
!cat PhoBERT-STL/results/two_stage_training/final_report.txt

---
## 🔵 Model 5/6: BiLSTM-MTL (Multi-Task Learning)

- **Architecture**: PhoBERT Embeddings (frozen) → BiLSTM → MultiHead Self-Attention → CNN → 2 Task Heads
- **Tokenizer**: `5CD-AI/visobert-14gb-corpus` (chỉ dùng tokenize)
- **Epochs**: 70 (early stopping patience 7) | **Batch size**: 32 | **LR**: 3e-4
- **Wandb run**: `BiLSTM-MTL`
- **⏱️ ~40 phút**

In [ ]:
!{ACTIVATE} && MPLBACKEND=Agg python BILSTM-MTL/train_bilstm_mtl.py --config BILSTM-MTL/config_bilstm_mtl.yaml

In [ ]:
# Xem kết quả BiLSTM-MTL
!cat BILSTM-MTL/models/mtl/final_report.txt 2>/dev/null || echo "Report file not found at expected path"

---
## 🟢 Model 6/6: BiLSTM-STL (Single-Task Learning - 2 Stage)

- **Architecture**: PhoBERT Embeddings (frozen) → BiLSTM → MultiHead Self-Attention → CNN → Sequential Heads
- **Tokenizer**: `5CD-AI/visobert-14gb-corpus` (chỉ dùng tokenize)
- **Epochs**: AD 70, SC 70 (early stopping patience 5-7) | **Batch size**: 32 | **LR**: 3e-4
- **Wandb runs**: `BiLSTM-STL-AD`, `BiLSTM-STL-SC`
- **⏱️ ~40 phút** (2 stages, with early stopping)

In [ ]:
!{ACTIVATE} && MPLBACKEND=Agg python BILSTM-STL/train_two_stage_bilstm.py --config BILSTM-STL/config_bilstm_stl.yaml

In [ ]:
# Xem kết quả BiLSTM-STL
!cat BILSTM-STL/results/two_stage_training/final_report.txt

---
---
# 📊 TỔNG HỢP KẾT QUẢ

In [ ]:
import json
import os

print("=" * 80)
print("TỔNG HỢP KẾT QUẢ HUẤN LUYỆN 6 MÔ HÌNH ABSA")
print("=" * 80)

result_paths = {
    "ViSoBERT-MTL": "VisoBERT-MTL/models/mtl/test_results.json",
    "ViSoBERT-STL": "VisoBERT-STL/models/sentiment_classification/test_results.json",
    "PhoBERT-MTL":  "phoBERT-MTL/models/mtl/test_results.json",
    "PhoBERT-STL":  "PhoBERT-STL/models/sentiment_classification/test_results.json",
    "BiLSTM-MTL":   "BILSTM-MTL/models/mtl/test_results.json",
    "BiLSTM-STL":   "BILSTM-STL/models/sentiment_classification/test_results.json",
}

ad_result_paths = {
    "ViSoBERT-STL": "VisoBERT-STL/models/aspect_detection/test_results.json",
    "PhoBERT-STL":  "PhoBERT-STL/models/aspect_detection/test_results.json",
    "BiLSTM-STL":   "BILSTM-STL/models/aspect_detection/test_results.json",
}

header = f"{'Model':<16} {'AD Acc':>8} {'AD F1':>8} {'SC Acc':>8} {'SC F1':>8}"
print(f"\n{header}")
print("-" * 52)

for model_name, sc_path in result_paths.items():
    ad_acc = ad_f1 = sc_acc = sc_f1 = "N/A"
    
    if os.path.exists(sc_path):
        with open(sc_path) as f:
            data = json.load(f)
            
            if "MTL" in model_name:
                if "ad" in data and isinstance(data["ad"], dict):
                    ad_section = data["ad"]
                    ad_acc = f"{ad_section.get('test_accuracy', 0)*100:.2f}%"
                    ad_f1 = f"{ad_section.get('test_f1', 0)*100:.2f}%"
                    sc_section = data.get("sc", {})
                    sc_acc = f"{sc_section.get('test_accuracy', 0)*100:.2f}%"
                    sc_f1 = f"{sc_section.get('test_f1', 0)*100:.2f}%"
                else:
                    ad_acc = f"{data.get('ad_accuracy', data.get('test_ad_accuracy', 0))*100:.2f}%"
                    ad_f1 = f"{data.get('ad_f1', data.get('test_ad_f1', 0))*100:.2f}%"
                    sc_acc = f"{data.get('sc_accuracy', data.get('test_sc_accuracy', 0))*100:.2f}%"
                    sc_f1 = f"{data.get('sc_f1', data.get('test_sc_f1', 0))*100:.2f}%"
            else:
                sc_acc = f"{data.get('test_accuracy', data.get('accuracy', 0))*100:.2f}%"
                sc_f1 = f"{data.get('test_f1', data.get('f1', 0))*100:.2f}%"
    
    if model_name in ad_result_paths:
        ad_path = ad_result_paths[model_name]
        if os.path.exists(ad_path):
            with open(ad_path) as f:
                ad_data = json.load(f)
                ad_acc = f"{ad_data.get('test_accuracy', ad_data.get('accuracy', 0))*100:.2f}%"
                ad_f1 = f"{ad_data.get('test_f1', ad_data.get('f1', 0))*100:.2f}%"
    
    print(f"{model_name:<16} {ad_acc:>8} {ad_f1:>8} {sc_acc:>8} {sc_f1:>8}")

print("\n" + "=" * 80)
print("AD = Aspect Detection | SC = Sentiment Classification")
print("Acc = Accuracy | F1 = F1 Score (macro average)")
print("=" * 80)

---
## 💾 Lưu kết quả

Lưu toàn bộ model weights và kết quả vào output của Kaggle.

In [ ]:
import shutil
import os

output_dir = "/kaggle/working/ABSA-results"
os.makedirs(output_dir, exist_ok=True)

model_dirs = {
    "ViSoBERT-MTL": ["VisoBERT-MTL/models/mtl"],
    "ViSoBERT-STL": ["VisoBERT-STL/models/aspect_detection", "VisoBERT-STL/models/sentiment_classification", "VisoBERT-STL/results"],
    "PhoBERT-MTL":  ["phoBERT-MTL/models/mtl"],
    "PhoBERT-STL":  ["PhoBERT-STL/models/aspect_detection", "PhoBERT-STL/models/sentiment_classification", "PhoBERT-STL/results"],
    "BiLSTM-MTL":   ["BILSTM-MTL/models/mtl"],
    "BiLSTM-STL":   ["BILSTM-STL/models/aspect_detection", "BILSTM-STL/models/sentiment_classification", "BILSTM-STL/results"],
}

for model_name, dirs in model_dirs.items():
    for src_dir in dirs:
        if os.path.exists(src_dir):
            dst_dir = os.path.join(output_dir, src_dir)
            os.makedirs(os.path.dirname(dst_dir), exist_ok=True)
            if os.path.exists(dst_dir):
                shutil.rmtree(dst_dir)
            shutil.copytree(src_dir, dst_dir)
            print(f"✅ Copied: {src_dir}")
        else:
            print(f"⚠️ Not found: {src_dir}")

print(f"\n📦 Tất cả kết quả đã được lưu tại: {output_dir}")

---
## 📊 Kiểm định McNemar (Statistical Testing)

Thực hiện kiểm định xem sự khác biệt giữa các mô hình có ý nghĩa thống kê hay không.

In [ ]:
!{ACTIVATE} && python scripts/run_mcnemar_test.py

## 📑 Tự động tạo Bảng Báo Cáo (LaTeX Tables)

Trích xuất F1/Accuracy từ eval metrics và render LaTeX table cho luận văn.

In [ ]:
!{ACTIVATE} && python scripts/generate_thesis_tables.py

---
## ✅ Hoàn tất!

**Tất cả 6 mô hình đã được huấn luyện.** Kết quả được lưu tại `/kaggle/working/ABSA-results/`.

📊 **Wandb Dashboard**: Xem chi tiết tại [wandb.ai](https://wandb.ai) → project **ABSA-Vietnamese**

Các file quan trọng:
- `best_model.pt` — Model weights tốt nhất
- `test_results.json` — Kết quả đánh giá trên test set
- `training_history.csv` — Lịch sử training
- `confusion_matrix_*.png` — Ma trận nhầm lẫn
- `final_report.txt` — Báo cáo chi tiết